# Stellar Astrophysics (Academic Year 2025/2026)



**Lectures:** Maurizio Giannotti <br>
**Tutorials:** Francisco Rodríguez Candón (francandon@unizar.es) and Mathieu Kaltschmidt (mkaltschmidt@unizar.es)

---

## Goal: build a model of the Sun with MESA, and check that it really is the Sun

You have run (or are running) the MESA setup in `Sun/`: a 1 M$_\odot$ star with $Z = 0.02$, from the
pre-main sequence to an age of 4.5 Gyr. This notebook goes through its output step by step. At every step we
compare one number of the model with the real Sun and store it in the dictionary `check`; the last step
puts all of them in one table.

**The Sun part is complete:** run the cells one by one, read them, and answer the questions on the way. Then, in
**Part 3**, it is your turn: you write the code for a 15 M$_\odot$ star yourself, using the Sun cells as templates.

We read the MESA output with the python package **`mesa_reader`**
([docs](https://billwolf.space/py_mesa_reader/)), `pip install mesa_reader`. Run the notebook from the
`MESA_Tutorial/` folder. If your own run has not finished, use the output already in the repository:

| folder | what it is |
|---|---|
| `Sun/LOGS/` | 1 M$_\odot$, $Z = 0.02$, pre-main sequence to 4.5 Gyr |
| `Star_15M/LOGS/` | 15 M$_\odot$, $Z = 0.02$, from the pre-main sequence until helium is exhausted in the core (for the last part) |

In [ ]:
# Import relevant modules
import shutil

import numpy as np
import matplotlib.pyplot as plt
import mesa_reader as mr

# Plot style: serif fonts, ticks inwards on all four sides. Labels use LaTeX if it is installed
# (if you get LaTeX errors, set USE_TEX = False).
USE_TEX = shutil.which("latex") is not None
plt.rcParams.update({
    "text.usetex": USE_TEX, "font.family": "serif", "mathtext.fontset": "cm",
    "axes.unicode_minus": False, "axes.linewidth": 0.8,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.top": True, "ytick.right": True,
    "axes.labelsize": 12, "axes.titlesize": 12,
    "xtick.labelsize": 10, "ytick.labelsize": 10,
})
if not USE_TEX:
    plt.rcParams.update({"font.serif": ["cmr10"], "axes.formatter.use_mathtext": True})

LOGS_SUN = "./Sun/LOGS"      # <-- change to "path/to/your/Sun/LOGS" if you ran it yourself
LOGS_15  = "./Star_15M/LOGS"

check = {}                   # model values to compare with the real Sun at the end

## Part 1: the history file, how the star changes with time

MESA writes two kinds of files into `LOGS/`:

* **`history.data`**: one line per saved time step: age, luminosity, $T_{\rm eff}$, central temperature, … → *how the star changes with time*
* **`profileN.data`**: one file per saved model: $T$, $\rho$, composition, … in every shell of the star → *what the star looks like inside at that moment*

In [ ]:
h = mr.MesaData(LOGS_SUN + "/history.data")

print("Number of saved models:", len(h.model_number))
print("Columns:", h.bulk_names)

Every column is an array with one value per saved model: `h.star_age`, `h.log_L`, … The last model, the Sun
today, is index `-1`.

In [ ]:
final_age = h.star_age[-1]          # yr
print(f"Final age: {final_age/1e9:.2f} Gyr   (max_age in inlist_project)")

check["age [Gyr]"] = final_age / 1e9

Luminosity against effective (surface) temperature. By tradition the temperature axis runs **backwards**:
hot stars on the left.

In [ ]:
plt.figure(figsize=(6, 4.5))
plt.plot(h.log_Teff, h.log_L, color="C3", lw=1.2, label=r"1 $M_\odot$ model")
plt.plot(h.log_Teff[-1], h.log_L[-1], "*", ms=10, color="C3", label="last model (4.5 Gyr)")
plt.plot(np.log10(5772), 0, marker=r"$\odot$", ms=12, color="k", ls="none", label="the real Sun")
plt.gca().invert_xaxis()
plt.xlabel(r"$\log\,T_{\rm eff}$ [K]")
plt.ylabel(r"$\log\,L$ [$L_\odot$]")
plt.legend(fontsize=9, loc="lower left")
plt.title("HR diagram")
plt.show()

check["Teff [K]"] = 10**h.log_Teff[-1]
check["L [Lsun]"] = 10**h.log_L[-1]
print(f"Today: Teff = {check['Teff [K]']:.0f} K,  L = {check['L [Lsun]']:.3f} Lsun")

**Question:** the track starts at the top right and first moves *down* almost vertically, then turns left
onto the main sequence. What is the star doing in each of those two phases? (Hint: `create_pre_main_sequence_model = .true.` in the inlist.)

In [ ]:
radius = 10**h.log_R

plt.figure(figsize=(6, 4.5))
plt.plot(h.star_age/1e9, radius, color="C0", lw=1.2)
plt.xlabel("age [Gyr]")
plt.ylabel(r"$R$ [$R_\odot$]")
plt.title("radius")
plt.show()

check["R [Rsun]"] = radius[-1]
print(f"Radius today: {radius[-1]:.3f} Rsun")

The history columns of **central** quantities start with `log_cntr_` and `center_` (see the list of Step 1).

In [ ]:
Tc = 10**h.log_cntr_T / 1e6
Xc = h.center_h1

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 5.5), sharex=True)
ax1.plot(h.star_age/1e9, Tc, color="C3", lw=1.2)
ax1.set_ylabel(r"$T_c$ [$10^6$ K]")
ax1.set_ylim(0, 17)
ax2.plot(h.star_age/1e9, Xc, color="C0", lw=1.2)
ax2.set_ylabel(r"$X_c$")
ax2.set_ylim(0, 0.8)
ax2.set_xlabel("age [Gyr]")
ax1.set_title("centre")
plt.show()

check["Tc [MK]"] = Tc[-1]
check["X_c"] = Xc[-1]
print(f"Today: Tc = {Tc[-1]:.2f} million K,  X_c = {Xc[-1]:.3f}  (it started at {Xc[0]:.2f})")

**Question:** hydrogen burns into helium in the centre, and the central temperature *rises* slowly. Why?
(Hint: four protons and four electrons become one He nucleus and two electrons: what happens to the number of
particles, and therefore to the pressure?)

## Part 2: the profile files, what the star looks like inside

In [ ]:
logs = mr.MesaLogDir(LOGS_SUN)
print("Profiles saved at models:", logs.model_numbers)

p = logs.profile_data()             # no argument: the LAST profile, i.e. the Sun today
print(f"Model {int(p.header('model_number'))}, age {p.header('star_age')/1e9:.2f} Gyr, {len(p.zone)} shells")
print("Columns:", p.bulk_names)

Two things to know about profiles:

* MESA numbers the shells from the **surface** (index `0`) to the **centre** (index `-1`).
* `mesa_reader` un-logs columns for you: the file has `logT`, `logRho`, `logR`, but `p.T`, `p.Rho`, `p.R` also work.

In [ ]:
R = p.R      # radius [Rsun]
T = p.T      # temperature [K]

plt.figure(figsize=(6, 4.5))
plt.plot(R, T, color="C3", lw=1.2)
plt.yscale("log")
plt.xlabel(r"$r$ [$R_\odot$]")
plt.ylabel(r"$T$ [K]")
plt.title("temperature")
plt.show()

print(f"Centre: {T[-1]/1e6:.2f} million K,  surface: {T[0]:.0f} K")

The same plot for the density (column `Rho`, in g cm$^{-3}$).

In [ ]:
rho = p.Rho

plt.figure(figsize=(6, 4.5))
plt.plot(R, rho, color="k", lw=1.2)
plt.yscale("log")
plt.xlabel(r"$r$ [$R_\odot$]")
plt.ylabel(r"$\rho$ [g cm$^{-3}$]")
plt.title("density")
plt.show()

check["rho_c [g/cm3]"] = rho[-1]
print(f"Centre: {rho[-1]:.1f} g/cm3,  surface: {rho[0]:.1e} g/cm3")

`x_mass_fraction_H`, `y_mass_fraction_He` and `z_mass_fraction_metals` are the mass fractions $X$, $Y$, $Z$
($X + Y + Z = 1$). The columns `h1`, `he3`, `he4`, `c12`, … are the individual isotopes.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

ax1.plot(R, p.x_mass_fraction_H, color="C0", lw=1.2, label=r"$X$ (H)")
ax1.plot(R, p.y_mass_fraction_He, color="C1", lw=1.2, ls="--", label=r"$Y$ (He)")
ax1.set_xlabel(r"$r$ [$R_\odot$]")
ax1.set_ylabel("mass fraction")
ax1.legend(fontsize=9)

isotopes = ["h1", "he4", "he3", "c12", "n14", "o16"]
for iso in isotopes:
    ax2.plot(p.mass, p.data(iso), lw=1.2, label=iso)
ax2.set_yscale("log")
ax2.set_ylim(1e-6, 1.5)
ax2.set_xlabel(r"$m$ [$M_\odot$]")
ax2.set_ylabel("mass fraction")
ax2.legend(fontsize=9, ncol=2, loc="lower right")
plt.show()

**Questions**

1. Why is there less H and more He in the centre than at the surface? The Sun started with the same composition everywhere.
2. Why is $^{3}$He *more* abundant at $m \approx 0.5\,M_\odot$ than at the centre, where it is produced fastest?
3. Where does $^{12}$C drop and $^{14}$N rise, and what does that tell you about the CNO cycle in the Sun?



`eps_nuc` is the nuclear energy released per gram and per second in each shell [erg g$^{-1}$ s$^{-1}$], and
`dm` is the mass of each shell [g]. Their product summed over all the shells is the total nuclear power of the
star. A star in equilibrium radiates from its surface exactly what it burns in its core.

In [ ]:
Lsun = 3.828e33               # erg/s (the value MESA uses)

L_nuc = np.sum(p.eps_nuc * p.dm) / Lsun

check["L_nuc / L"] = L_nuc / 10**h.log_L[-1]
print(f"nuclear power: {L_nuc:.4f} Lsun,  luminosity at the surface: {10**h.log_L[-1]:.4f} Lsun")

## Step 11. Is it the Sun?

All the numbers you collected, against the real Sun (observations and the standard solar model).

In [ ]:
real_sun = {"age [Gyr]": 4.57, "Teff [K]": 5772, "L [Lsun]": 1.0, "R [Rsun]": 1.0,
            "Tc [MK]": 15.7, "rho_c [g/cm3]": 150, "X_c": 0.34, "L_nuc / L": 1.0}

print(f"{'quantity':<15}{'model':>10}{'real Sun':>10}{'model/Sun':>11}")
for name, sun in real_sun.items():
    if name in check:
        print(f"{name:<15}{check[name]:>10.4g}{sun:>10.4g}{check[name]/sun:>11.3f}")
    else:
        print(f"{name:<15}{'(step not done yet)':>31}")

Not bad for a few lines changed in the inlist! Things we did **not** tune: the mixing-length parameter, the
initial helium abundance, element diffusion. Real solar models adjust these so that $R$, $L$ and the surface
$Z/X$ match exactly at 4.57 Gyr.

---

## Part 3, on your own: a 15 M$_\odot$ star

The *minimal example* you ran at the start (`$MESA_DIR/star/work`) is a 15 M$_\odot$ star that stops as soon as
it reaches the main sequence. `Star_15M/inlist_project_solution` takes it much further: it lets the star burn all the
hydrogen and then all the helium in its core, with mass loss switched on once it becomes a cool supergiant. The
output of that run is in `Star_15M/LOGS/`.

Use what you did for the Sun. The history `h15` and the last profile `p15` are loaded below; they have the same
columns as the Sun's.

1. **HR diagram** of both stars on the same plot. Where is the 15 M$_\odot$ star at the end? How big is it?
2. **Mass:** the star started with 15 M$_\odot$. How much is left (`star_mass`), and where did the rest go?
3. **Centre:** plot $\log T_c$ against $\log \rho_c$ for both stars. How much hotter and denser is the centre of the 15 M$_\odot$ star at the end?
4. **Inside:** plot the H, He, C and O mass fractions of the last profile against the fractional mass $m/M$. What are the layers of this star, from the centre outwards?
5. **Time:** how long did the main sequence last ($X_c$ from its initial value to $\approx 0$), and how long core helium burning? Compare with the Sun.

In [ ]:
h15  = mr.MesaData(LOGS_15 + "/history.data")
p15  = mr.MesaLogDir(LOGS_15).profile_data()

print(f"15 Msun: age {h15.star_age[0]/1e6:.1f} to {h15.star_age[-1]/1e6:.1f} Myr")

In [ ]:
# 1. HR diagram of both stars

In [ ]:
# 2. Mass lost

In [ ]:
# 3. Central temperature against central density

In [ ]:
# 4. Composition of the last profile

In [ ]:
# 5. Duration of core H and core He burning

**To think about.** The 15 M$_\odot$ core ends up about 18 times hotter and 30 times denser than the Sun's.
Nuclear rates and neutrino losses are extremely steep functions of $T$, and so is the emission of any
hypothetical light, weakly coupled particle (axions!). That is why massive stars, not the Sun, are the
laboratory used in the "home-made example" of the lecture: a small extra energy loss from the core changes how
fast these stars evolve.

## Where to go from here

* Change one thing in the Sun's inlist and see what happens: `initial_mass = 2`, or `initial_z = 0.001`, or `use_gold_tolerances = .true.` (compare the run time).
* MESA ships with ~100 ready-made setups in `$MESA_DIR/star/test_suite/` (white dwarfs, massive stars to core collapse, binaries …). Copy one, read its `README.rst`, run it.
* Documentation: https://docs.mesastar.org, and the `mesa-users` mailing list, where the developers answer questions daily.